# Roast my image

**Difficulty:** Beginner

In [ ]:
// Paths resolve from this notebook's folder, like the imports.
const photoBytes = await Deno.readFile("../../assets/images/IMG_1494.jpg");

// Show the exact bytes the request below sends.
Deno.jupyter.image(photoBytes);

In [ ]:
import { choice, image, noul, score } from "npm:@typesafe-ai/sdk@^0.7.0";
import { client } from "../../assets/typesafe-client.ts";

const DANGER_ZONE = { headers: { "TypeSafe-Danger-Zone-Images": "true" } };

const photo = image(photoBytes);

const result = await client.systemOne(
  {
    state: { photo },
    questions: {
      synthetic_bokeh_plausibility: score("with `photo`, how consistently does background blur follow the depth of the scene?", [
        "The blur boundary cuts through the subject: hair, glasses arms, ear edges or held objects are blurred while adjacent background is sharp, or a halo of sharp background hugs the outline.",
        "Depth is mostly coherent, but one region is wrong: e.g. a gap between arm and torso keeps sharp background, or a foreground object at subject distance is blurred.",
        "Blur increases with distance and the ground plane falls off gradually; the only uncertainty is in fine hair.",
        "Foreground and background blur both scale with distance, out-of-focus highlights are consistent discs, and no outline artefacts are visible.",
      ]),
      generative_detail_coherence: score("With `photo`, does fine detail (text, hands, repeated structures, foliage, fabric) hold together as real objects when examined?", [
        "Pseudo-text with glyph-like shapes, fused or extra fingers, or structures (railings, windows, teeth) that merge or change count.",
        "One area where detail dissolves into plausible texture that isn't any particular object, e.g. a painted-looking crowd or foliage.",
        "Detail is coherent wherever inspected but carries a uniform waxy or over-defined micro-texture, typical of AI denoise or upscaling.",
        "Fine detail behaves like optical capture: text is legible or uniformly blurred, repeated objects vary naturally, and grain is consistent across regions.",
      ]),
      lighting_physics_consistency: score("With `photo`, do shadows, highlights and reflections across the whole frame agree on where the light comes from?", [
        "Contradictory: shadows from objects in the same sunlight point in clearly different directions, or a reflection is missing or shows a different scene.",
        "One localised mismatch: one element (sky, person, object) is lit from a different direction or with a different colour than its surroundings, or sits without a contact shadow.",
        "Consistent overall, with an anomaly explainable by a second visible or plausible source (window, flash).",
        "Every shadow, highlight and reflection checked is consistent with the visible light sources.",
      ]),
      origin_triage: score("With `photo`, how likely is it that the image is AI-generated or heavily edited?", [
        "Clearly synthetic or heavily edited: obvious artifacts, unnatural textures, or impossible geometry.",
        "Some signs of synthetic origin or heavy editing, but not immediately obvious.",
        "Mostly consistent with real-world capture, minor anomalies could be due to editing.",
        "No detectable signs of synthetic origin or heavy editing; looks like a genuine photograph.",
      ]),
      composition: score("How well composed is `photo`?", ["poor", "okay", "great"]),
      biggest_problem: choice("What is the biggest problem with `photo`?", {
        lighting: "Too dark, too bright, or flat lighting.",
        composition: "Awkward framing or no clear subject.",
        focus: "Blurry or out of focus.",
        clutter: "Too much going on.",
        none: "Nothing stands out.",
      }),
    },
  },
  DANGER_ZONE,
);

console.log(JSON.stringify(result.answers, null, 2));
console.log("input tokens, image included:", result.usage.input_tokens);